# 02 — Evaluate and select (many runs → few registrations)

Review MLflow experiment runs and **select only the best** for the OpenShift AI Model Registry.

Important demo point: weaker runs stay in MLflow for lineage; they are **not** registered.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
    os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from src.mlflow_utils import configure_tracking, pick_best_run, search_runs_table

configure_tracking()
rows = search_runs_table()
rows

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
if df.empty:
    raise SystemExit("No MLflow runs found. Run notebooks/01_train_track.ipynb first.")

display_cols = [c for c in ["run_id", "run_name", "mAP50", "mAP50-95", "precision", "recall", "epochs", "imgsz", "lr0", "status"] if c in df.columns]
df[display_cols].sort_values(by="mAP50", ascending=False, na_position="last")

## Selection policy

Default: highest `mAP50`. Override by setting `SELECTED_RUN_ID` in the environment, or edit `selected` below.

In [ ]:
override = os.environ.get("SELECTED_RUN_ID")
if override:
    matches = [r for r in rows if r["run_id"] == override]
    if not matches:
        raise SystemExit(f"SELECTED_RUN_ID={override} not found in experiment")
    selected = matches[0]
else:
    selected = pick_best_run(rows, metric="mAP50")

if not selected:
    raise SystemExit("Could not pick a best run (missing mAP50 metrics?)")

rejected = [r for r in rows if r["run_id"] != selected["run_id"]]
print("SELECTED for registration candidate:")
print(selected)
print()
print(f"NOT registering {len(rejected)} other run(s) — they remain experiment history only:")
for r in rejected:
    print(f"  - {r.get('run_name')} ({r['run_id']}) mAP50={r.get('mAP50')}")

os.environ["SELECTED_RUN_ID"] = selected["run_id"]
print()
print("Exported SELECTED_RUN_ID=" + selected["run_id"])
print("Next: notebooks/03_export_register.ipynb")